# 01 —


In [22]:
pip install requests   

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Task 1.1 — Read with the csv module

In [23]:
import csv, json

with open("data/appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    rows = list(reader)
print("Row count:", len(rows))

def parse_row(row):
    out = dict(row)
    out["age"] = int(row["age"])
    out["fee_inr"] = int(row["fee_inr"])
    out["prior_no_shows"] = int(row["prior_no_shows"])
    out["distance_km"] = float(row["distance_km"])
    return out

accepted, rejected = [], []
for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)
print("Accepted:", len(accepted), "Rejected:", len(rejected))

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted: 988 Rejected: 32


## Task 1.2 

In [24]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/appointments.csv")
print(df.shape)
df.info()
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print("Raw booking_channel spellings:", df["booking_channel"].nunique())

df = df.drop_duplicates()
print("After (a) drop duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("After (b) clean booking_channel:", len(df), "| distinct:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After (c) date conversion:", len(df))

df = df[df["appt_date"].notna() & df["status"].notna()]
print("After (d) drop missing appt_date / status:", len(df))

(1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_dat

## Task 1.3


In [25]:
df = df.copy()
bad_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", bad_age.sum())
df.loc[bad_age, "age"] = np.nan
print("Missing ages in total:", df["age"].isnull().sum())

median_age = df["age"].median()
median_dist = df["distance_km"].median()
print("Median age:", median_age, "| median distance:", median_dist)
df["age"] = df["age"].fillna(median_age).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(median_dist)

for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    capped = ((df[col] < low) | (df[col] > high)).sum()
    df[col] = np.clip(df[col], low, high)
    print(col, "Q1", q1, "Q3", q3, "lower fence", low, "upper fence", high, "capped", capped)

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest fee:", df["fee_inr"].max())

Invalid ages: 5
Missing ages in total: 17
Median age: 48.0 | median distance: 8.25
distance_km Q1 4.625 Q3 13.0 lower fence -7.9375 upper fence 25.5625 capped 26
fee_inr Q1 500.0 Q3 787.5 lower fence 68.75 upper fence 1218.75 capped 3
Largest fee: 1219


## Task 1.4 

In [26]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print("No-shows:", df["no_show"].sum(), "of", len(df), "=", round(100 * df["no_show"].mean(), 1), "%")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("Column means:", X.mean(axis=0).round(2))

mask = X[:, 3] > 14
print("lead_days > 14:", mask.sum(), "rows,", round(100 * mask.mean(), 1), "%")

x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

lead_days range: 1 to 45
No-shows: 218 of 990 = 22.0 %
Column means: [ 48.17   9.5  635.71  11.82]
lead_days > 14: 296 rows, 29.9 %
Rows with |z| > 2 for distance_km: 53


## Task 1.5

In [27]:
import requests

clinics = pd.read_csv("data/clinics.csv")
temps = []
for _, c in clinics.iterrows():
    temp, reason = None, ""
    for attempt in range(2):
        try:
            r = requests.get("https://api.open-meteo.com/v1/forecast",
                             params={"latitude": c["latitude"], "longitude": c["longitude"],
                                     "hourly": "temperature_2m", "forecast_days": 1}, timeout=10)
            if r.status_code != 200:
                reason = f"status {r.status_code}"
                continue
            hourly = r.json().get("hourly")
            if not hourly:
                reason = "hourly missing"
                continue
            temp = hourly["temperature_2m"][0]
            break
        except requests.exceptions.RequestException as e:
            reason = str(e)[:80]
    if temp is None:
        print(c["clinic_name"], "-> temp_c stored as None:", reason)
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics)

  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.9  
1    18.5  
2    19.9  
3    17.0  
4    17.0  
5    25.1  
6    19.1  
7    15.1  


## Task 1.6

In [28]:
df = df.sort_values("appointment_id")
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
cols = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender", "distance_km",
        "fee_inr", "booking_channel", "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"]
df = df[cols]

print("appointment_id unique:", df["appointment_id"].is_unique, "| distinct:", df["appointment_id"].nunique())
print("Missing values in the 16 columns:", df.isnull().sum().sum())
df.to_csv("appointments_clean.csv", index=False)
print("appointments_clean.csv:", df.shape)

ce = pd.read_csv("clinics_enriched.csv")
with open("load_data.sql", "w") as f:
    for _, c in ce.iterrows():
        temp = "NULL" if pd.isna(c["temp_c"]) else c["temp_c"]
        f.write(f"INSERT INTO clinics VALUES ('{c['clinic_id']}','{c['clinic_name']}','{c['city']}','{c['tier']}',{c['latitude']},{c['longitude']},{temp});\n")
    for _, a in df.iterrows():
        f.write(f"INSERT INTO appointments VALUES ('{a['appointment_id']}','{a['patient_id']}','{a['clinic_id']}','{a['booked_on']}','{a['appt_date']}',{a['age']},'{a['gender']}',{a['distance_km']},{a['fee_inr']},'{a['booking_channel']}','{a['visit_type']}','{a['reminder_sent']}',{a['prior_no_shows']},'{a['status']}',{a['lead_days']},{a['no_show']});\n")
print("INSERT statements:", len(ce) + len(df))

appointment_id unique: True | distinct: 990
Missing values in the 16 columns: 0
appointments_clean.csv: (990, 16)
INSERT statements: 998


## Task 2.2 


In [29]:
chk = pd.read_csv("appointments_clean.csv").merge(ce, on="clinic_id")
print(chk.groupby(["clinic_name", "city"])["no_show"].agg(appointments="count", no_shows="sum"))

                                  appointments  no_shows
clinic_name              city                           
Edappally Medical Centre Kochi             142        23
Gomti Nagar Care Point   Lucknow            90        30
Hadapsar Care Centre     Pune              143        39
Kothrud Family Clinic    Pune              160        36
Malviya Nagar Clinic     Jaipur            142        38
Sitabuldi Health Point   Nagpur             82        13
Vaishali Nagar Clinic    Jaipur            130        18
Vijay Nagar Clinic       Indore            101        21
